### Links and Resources
| Concept / Command | Description |
|---|---|
| [`foreachBatch`](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#using-foreach-and-foreachbatch) | Run any batch operation on each micro-batch |
| [`foreach`](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#using-foreach-and-foreachbatch) | Row-level custom sink |
| [Delta `MERGE`](https://docs.delta.io/latest/delta-update.html#upsert-into-a-table-using-merge) | Idempotent upsert — the foundation of exactly-once `foreachBatch` |
| [Idempotent foreachBatch](https://docs.databricks.com/en/structured-streaming/delta-lake.html#idempotent-table-writes-in-foreachbatch) | Making foreachBatch exactly-once with a batch ID |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel — 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()


def display(df, n=20):
    df.show(n, truncate=False)

# `foreachBatch` and Custom Sinks

The standard streaming sinks — Delta, Kafka, files — cover most use cases. When they don't, **`foreachBatch`** is the escape hatch.

`foreachBatch` hands you a **regular (non-streaming) DataFrame** for each micro-batch.  Inside the function, anything goes: `count()`, `collect()`, `MERGE`, HTTP calls, multi-sink writes.

```python
def process_batch(batch_df, batch_id):
    # batch_df is a plain DataFrame — not streaming
    n = batch_df.count()          # legal here
    batch_df.write.save(...)       # write anywhere
    batch_df.sparkSession.sql("MERGE INTO ...") # SQL operations

stream.writeStream.foreachBatch(process_batch).start()
```

**When to use `foreachBatch`:**
- Upsert (MERGE) a Delta table — the standard sink only appends.
- Write the same batch to multiple sinks (fan-out).
- Call an external API or service per batch.
- Run aggregations inside the batch that streaming doesn't support natively.
- Apply complex business logic that requires batch-mode DataFrames.

## 1. The basics — inspect each micro-batch

In [ ]:
import time, uuid
from pyspark.sql import functions as F

CKPT_BASE = "/Volumes/population_metrics/landing/datasets/streaming_demo/_checkpoints"

# rate-micro-batch (not plain rate) so availableNow can terminate the stream — see notebook 02.
events = (spark.readStream
    .format("rate-micro-batch").option("rowsPerBatch", 25).option("numPartitions", 1).load()
    .withColumnRenamed("timestamp", "event_time")
    .withColumnRenamed("value",     "event_id")
    .withColumn("user_id", (F.col("event_id") % 5).cast("int"))
    .withColumn("amount",  (F.rand() * 100 + 1).cast("int")))

batch_log = []

def inspect_batch(batch_df, batch_id):
    n     = batch_df.count()
    total = batch_df.agg(F.sum("amount")).collect()[0][0] or 0
    batch_log.append({"batch_id": batch_id, "rows": n, "total_amount": total})
    print(f"  batch {batch_id}: {n} rows, total_amount={total:.0f}")

q = (events.writeStream
     .foreachBatch(inspect_batch)
     .outputMode("append")
     .option("checkpointLocation", f"{CKPT_BASE}/inspect_batch_{uuid.uuid4().hex[:8]}")
     .trigger(availableNow=True)
     .start())

q.awaitTermination()

print(f"\nProcessed {len(batch_log)} batches")

## 2. Idempotence — why `foreachBatch` needs MERGE

If a micro-batch fails halfway through and is retried, `foreachBatch` will be called again with the **same `batch_id`** and the same rows.

| Sink operation | Safe on retry? | Why |
|---|---|---|
| `batch_df.write.mode("append").save(...)` | **No** — duplicates | Rows written twice |
| `INSERT INTO target ...` | **No** — duplicates | Same rows inserted again |
| `MERGE INTO target ON key ...` | **Yes** | Matching keys are updated, not duplicated |

**Rule:** make `foreachBatch` idempotent by keying the write on a business key (or `batch_id` for a coarser guarantee).

## 3. MERGE — the idempotent upsert pattern

The canonical `foreachBatch` pattern for a Delta target:

```
  Each micro-batch:
  1. Create a temp view from batch_df
  2. MERGE INTO target ON business_key
     WHEN MATCHED    → UPDATE the row (no duplicate)
     WHEN NOT MATCHED → INSERT new row

  If the batch is re-delivered → MERGE ON key is safe: already-inserted rows
  are updated (idempotent), not duplicated.
```

In [ ]:
# ── Create the target table ───────────────────────────────────────────────────
spark.sql("DROP TABLE IF EXISTS population_metrics.default.user_totals")
spark.sql(
    "CREATE TABLE population_metrics.default.user_totals "
    "(user_id INT, total_amount DOUBLE, event_count LONG, last_seen TIMESTAMP) "
    "USING DELTA"
)

# ── foreachBatch with MERGE ───────────────────────────────────────────────────
def upsert_user_totals(batch_df, batch_id):
    if batch_df.isEmpty():
        return
    # Aggregate within the batch first (reduce rows before the MERGE)
    agg = (batch_df
           .groupBy("user_id")
           .agg(
               F.sum("amount").alias("batch_amount"),
               F.count("*").alias("batch_count"),
               F.max("event_time").alias("batch_last_seen"),
           ))
    agg.createOrReplaceTempView("upsert_src")

    batch_df.sparkSession.sql(
        "MERGE INTO population_metrics.default.user_totals AS t "
        "USING upsert_src AS s ON t.user_id = s.user_id "
        "WHEN MATCHED THEN UPDATE SET "
        "  t.total_amount = t.total_amount + s.batch_amount, "
        "  t.event_count  = t.event_count  + s.batch_count, "
        "  t.last_seen    = s.batch_last_seen "
        "WHEN NOT MATCHED THEN INSERT "
        "  (user_id, total_amount, event_count, last_seen) "
        "VALUES (s.user_id, s.batch_amount, s.batch_count, s.batch_last_seen)"
    )

q = (events.writeStream
     .foreachBatch(upsert_user_totals)
     .outputMode("update")
     .option("checkpointLocation", f"{CKPT_BASE}/upsert_user_totals_{uuid.uuid4().hex[:8]}")
     .trigger(availableNow=True)
     .start())

q.awaitTermination()

print("Per-user aggregation in target table (upserted, not appended):")
spark.sql(
    "SELECT user_id, total_amount, event_count, last_seen "
    "FROM population_metrics.default.user_totals ORDER BY user_id"
).show(truncate=False)

## 4. Multi-sink fan-out — write one batch to two destinations

`foreachBatch` is the only way to write a single micro-batch to **more than one sink**. The batch DataFrame is re-used (Spark caches it in memory for the duration of the function).

In [ ]:
CKPT_FANOUT = f"{CKPT_BASE}/fanout_{uuid.uuid4().hex[:8]}"

spark.sql("DROP TABLE IF EXISTS population_metrics.default.fanout_premium")
spark.sql("DROP TABLE IF EXISTS population_metrics.default.fanout_all")

def fan_out(batch_df, batch_id):
    batch_df.cache()    # cache so both writes read from memory, not recomputed

    # Sink 1: all events → raw table (append)
    (batch_df.write.format("delta").mode("append")
             .saveAsTable("population_metrics.default.fanout_all"))

    # Sink 2: premium users only → filtered table
    (batch_df
     .filter(F.col("user_id").isin([0, 2, 4]))   # simulate "premium" tier
     .write.format("delta").mode("append")
     .saveAsTable("population_metrics.default.fanout_premium"))

    batch_df.unpersist()

q = (events.writeStream
     .foreachBatch(fan_out)
     .option("checkpointLocation", CKPT_FANOUT)
     .trigger(availableNow=True)
     .start())

q.awaitTermination()

all_cnt     = spark.sql("SELECT COUNT(*) FROM population_metrics.default.fanout_all").collect()[0][0]
premium_cnt = spark.sql("SELECT COUNT(*) FROM population_metrics.default.fanout_premium").collect()[0][0]
print(f"fanout_all:     {all_cnt} rows  (all users)")
print(f"fanout_premium: {premium_cnt} rows  (users 0,2,4 only — ~60% of all)")

## 5. Using `batch_id` as an idempotency key

For sinks that don't support MERGE, record the last processed `batch_id` and skip re-delivery:

```python
def idempotent_write(batch_df, batch_id):
    last_written = get_last_batch_id_from_target()   # read from a control table
    if batch_id <= last_written:
        print(f"batch {batch_id} already written — skipping")
        return
    batch_df.write.mode("append").save(target_path)
    record_batch_id(batch_id)   # update control table
```

Delta handles this automatically via its transaction log — prefer Delta sinks over manual `batch_id` tracking when possible.

## 6. `foreach` — row-level custom sink

`foreach` calls your function once per row. Use it when you need to send each record to an external system (an HTTP endpoint, a non-Spark database) and cannot batch them.

```python
class RowWriter:
    def open(self, partition_id, epoch_id):
        # called once per partition — open connection here
        return True
    def process(self, row):
        # called once per row
        requests.post("https://api.example.com/events", json=row.asDict())
    def close(self, error):
        # called when partition finishes or fails
        pass

stream.writeStream.foreach(RowWriter()).start()
```

**Prefer `foreachBatch` over `foreach`.** `foreach` has no batching, so each row is a separate call — very slow for high-throughput streams. Use `foreachBatch` with batch HTTP calls or bulk-inserts instead.

In [ ]:
# Lightweight foreach demo — collect rows to a list (not for production!)
collected = []

class ListCollector:
    def open(self, partition_id, epoch_id):
        return True
    def process(self, row):
        collected.append({"user_id": row.user_id, "amount": row.amount})
    def close(self, error):
        pass

q = (events.select("user_id", "amount")
     .writeStream
     .foreach(ListCollector())
     .option("checkpointLocation", f"{CKPT_BASE}/foreach_demo_{uuid.uuid4().hex[:8]}")
     .trigger(availableNow=True)
     .start())

q.awaitTermination()

print(f"Collected {len(collected)} rows via foreach:")
for r in collected[:5]:
    print(f"  {r}")

## 7. Common `foreachBatch` mistakes

| Mistake | Problem | Fix |
|---|---|---|
| Using `spark.sql(...)` instead of `batch_df.sparkSession.sql(...)` | Closure captures outer `spark` — may use wrong session in some cluster modes | Always use `batch_df.sparkSession` |
| `batch_df.write.mode("append")` without idempotence | Duplicates on retry | Use MERGE on a business key |
| Not calling `batch_df.cache()` before multiple writes | Second write recomputes from source (wasteful or wrong) | `cache()` before fan-out, `unpersist()` after |
| Long blocking calls inside the function | Delays next trigger; may time out | Use async or move slow calls outside |
| Raising exceptions without handling | Query terminates; requires manual restart | Catch expected errors; use DLQ for bad rows |

## Takeaways

- `foreachBatch` gives you a regular (non-streaming) DataFrame per batch — enabling `count()`, `collect()`, MERGE, multi-sink writes, and any other batch operation.
- The same `batch_id` may be re-delivered after a failure. **Always write idempotently** — MERGE on a business key is the standard pattern.
- For fan-out to multiple sinks, `cache()` the batch DataFrame before the first write and `unpersist()` after the last.
- Use `batch_df.sparkSession.sql(...)` inside `foreachBatch`, not the outer `spark` variable.
- `foreach` is row-level — useful only for sending individual records to external systems. Prefer `foreachBatch` with batched API calls for throughput.
- Don't reach for `foreachBatch` when `.toTable("delta_table")` would work — the engine's built-in sink has better optimisations.